# LoRA fine-tuning Qwen3.5-4B на Text2SQL

Полный пайплайн: install → load model → train LoRA → merge → GGUF → download.

**Окружение:** Google Colab, Tesla T4 (16 GB VRAM) — бесплатный tier.

**Входные данные** (загрузить через `Files` ВРУЧНУЮ или подключить Google Drive):
- `train.jsonl` — выход `scripts/build_finetune_dataset.py`
- `val.jsonl` — то же

**Учебные нюансы по ходу:**
- Почему unsloth, а не plain peft.
- Зачем `apply_chat_template` именно от tokenizer'a, а не вручную.
- Как читать loss-кривую: что есть здоровый train, что overfit.
- GGUF-конвертация: почему q4_K_M по умолчанию.

**Время прогона:** ~40-60 мин на ~5k примеров за 2 эпохи.

## 0. Установка зависимостей

**Учебный нюанс:** `unsloth` — это уровень абстракции над `peft + transformers + bitsandbytes`. Он даёт ~2× speedup и ~40% меньше VRAM на T4 за счёт ручных Triton-кернелей под attention. Альтернатива — голый PEFT, работает, но медленнее.

Версии пинуем — unsloth ломается при mismatch с transformers.

In [ ]:
# # ═══════════════════════════════════════════════════════════════
# # Установка ML-стека под unsloth_zoo 2026.5.x
# # Целевые версии (точно совместимы):
# #   torch 2.10        (= Colab base, не апгрейдим)
# #   transformers 4.51.3
# #   trl ≥ 0.20
# #   datasets 3.4.x
# #   numpy < 2.5
# #
# # ВАЖНО: после выполнения этой ячейки → Runtime → Restart session
# # (НЕ "Disconnect and delete" — иначе потеряются /content/*.jsonl)
# # ═══════════════════════════════════════════════════════════════

# # 1. Откатить torch к 2.10 (мы случайно ушли на 2.11 в предыдущих попытках)
# #    Colab base image и так хочет 2.10 — это совпадение, конфликта не будет
# !pip install -q --upgrade --force-reinstall \
#     "torch>=2.10,<2.11" \
#     "numpy>=2.0,<2.5"

# # 2. ML-стек, версии в открытых диапазонах unsloth_zoo
# !pip install -q --upgrade --force-reinstall \
#     "transformers==4.51.3" \
#     "trl>=0.20,<0.24" \
#     "datasets>=3.4.1,<4.0" \
#     "accelerate>=0.34" \
#     "peft>=0.13" \
#     "bitsandbytes>=0.43"

# # 3. unsloth + unsloth_zoo одной парой (важно: оба!)
# !pip install -q --upgrade unsloth unsloth_zoo

# # 4. Самопроверка: pip resolver видит критические конфликты?
# print()
# print("=" * 70)
# print("--- pip check для нашего стека ---")
# !pip check 2>&1 | grep -iE "unsloth|transformers|trl|datasets|torch[^audio|^vision]|numpy|peft" || echo "✓ для НАШЕГО стека конфликтов нет"
# print()
# print("⚠  СЕЙЧАС: Runtime → Restart session   (НЕ delete!)")
# print("    После рестарта запусти СЛЕДУЮЩУЮ ячейку проверки версий.")
# print("=" * 70)


# ⚠ После выполнения → Runtime → Restart session  (НЕ "Delete runtime" — /content/*.jsonl потеряются)
#
# Стратегия: unsloth сам тянет совместимые версии transformers/trl/datasets.
# Пытаться пинить их вручную → ResolutionImpossible (проверено на практике).
# Единственное что нужно явно: unsloth_zoo (pip не тянет его как зависимость).

!pip install -q --upgrade pip
!pip install -q --upgrade unsloth unsloth_zoo

print("✓ install done — теперь Runtime → Restart session")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.2 MB/s eta 0:00:00
✓ install done — теперь Runtime → Restart session


In [ ]:
# Запусти ПОСЛЕ Runtime → Restart session
import torch
import unsloth, unsloth_zoo, transformers, trl, datasets, peft
import numpy as np

print("✓ все библиотеки импортированы")
print()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:           ", torch.cuda.get_device_name(0))
    print("VRAM (GB):     ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

print()
print(f"unsloth      = {unsloth.__version__}")
print(f"unsloth_zoo  = {unsloth_zoo.__version__}")
print(f"transformers = {transformers.__version__}")
print(f"trl          = {trl.__version__}")
print(f"datasets     = {datasets.__version__}")
print(f"peft         = {peft.__version__}")
print(f"torch        = {torch.__version__}")
print(f"numpy        = {np.__version__}")

# Минимальные проверки здравомыслия
assert torch.cuda.is_available(), "GPU не найден — проверь Runtime → Change runtime type → T4"
print()
print("✓ GPU есть — готов к загрузке модели")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
✓ все библиотеки импортированы

CUDA available: True
GPU:            NVIDIA A100-SXM4-80GB
VRAM (GB):      85.1

unsloth      = 2026.5.6
unsloth_zoo  = 2026.5.4
transformers = 5.5.0
trl          = 0.24.0
datasets     = 4.3.0
peft         = 0.19.1
torch        = 2.10.0+cu128
numpy        = 2.0.2

✓ GPU есть — готов к загрузке модели


## 1. Загрузка данных

Два варианта — выбери один:

**(A) Загрузить файлы вручную** через панель Files слева. Удобно для маленького датасета (<50 MB).

**(B) Подключить Google Drive.** Положи `train.jsonl` / `val.jsonl` в `MyDrive/text2sql_finetune/` и расшарь в Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# !cp /content/drive/MyDrive/MyDrive/text2sql_finetune/*.gz /content/
# !gunzip -kf /content/*.gz
# !ls -la /content/*.jsonl

!cp /content/drive/MyDrive/MyDrive/text2sql_finetune/*.jsonl /content/

In [ ]:
# !ls /content/drive/MyDrive/MyDrive/ | head -30

!ls /content/

drive  sample_data  train.jsonl  unsloth_compiled_cache  val.jsonl


In [ ]:
# Sanity-check данных
import json
with open("train.jsonl") as f:
    sample = [json.loads(line) for line in f][:3]
for ex in sample:
    msgs = ex["messages"]
    print(f"--- example ({ex.get('_meta', {}).get('task')}) ---")
    for m in msgs:
        prev = m['content'][:120].replace('\n', ' ')
        print(f"  [{m['role']:9s}] {prev}...")
    print()

--- example (sql) ---
  [system   ]  Ты экспертный Text2SQL ассистент с глубоким пониманием банковской аналитики. Твоя задача - точно преобразовывать запрос...
  [user     ]  Преобразуй следующий запрос в SQL: Запрос: What is the restaurant's name and ID located at Ocean Avenue, San Francisco?...
  [assistant] SELECT T2.label, T1.id_restaurant FROM location AS T1 INNER JOIN generalinfo AS T2 ON T1.id_restaurant = T2.id_restauran...

--- example (sql) ---
  [system   ]  Ты экспертный Text2SQL ассистент с глубоким пониманием банковской аналитики. Твоя задача - точно преобразовывать запрос...
  [user     ]  Преобразуй следующий запрос в SQL: Запрос: Give the genre and popularity of movies whose revenue is at least 120,000,00...
  [assistant] SELECT T3.genre_name, T1.popularity FROM movie AS T1 INNER JOIN movie_genres AS T2 ON T1.movie_id = T2.movie_id INNER JO...

--- example (sql) ---
  [system   ]  Ты экспертный Text2SQL ассистент с глубоким пониманием банковской аналитики. Твоя задача 

## 2. Загрузка базовой модели в 4-bit

**Учебный нюанс — память T4:**
- 4B параметров в fp16 = 8 GB. Уже впритык, без места на оптимизатор.
- 4B параметров в 4-bit (NF4 quantization) = ~2.5 GB.
- LoRA-адаптеры r=16: ~50 M обучаемых параметров. В fp16 = 100 MB. AdamW state = 200 MB.
- Активации при batch=2, seq=4096: ~3 GB.
- Итого ~6 GB peak — спокойно в 16 GB T4.

**Почему 4-bit, а не 8-bit:** базовая модель ЗАМОРОЖЕНА. Её точность не критична, потому что обучается только LoRA-«дельта». В 4-bit потери качества инференса <1%.

**Если Qwen3.5-4B недоступна:** замени `model_name` на `Qwen/Qwen2.5-4B` или `Qwen/Qwen3-4B-Base` — пайплайн не зависит от конкретной версии.

In [ ]:
# # === ДИАГНОСТИКА + НАСТРОЙКА HF_TOKEN ===
# import os, subprocess, shutil

# # 1) Подхватить токен из Colab Secrets (добавь через 🔑 слева: Name=HF_TOKEN)
# try:
#     from google.colab import userdata
#     token = userdata.get("HF_TOKEN")
#     if token:
#         os.environ["HF_TOKEN"] = token
#         print(f"✓ HF_TOKEN из Secrets: {token[:8]}...")
#     else:
#         print("⚠ HF_TOKEN в Secrets не найден — добавь его (huggingface.co/settings/tokens)")
# except Exception:
#     print("⚠ не удалось прочитать Secrets — задай вручную: os.environ['HF_TOKEN'] = 'hf_...'")

# # 2) Сеть
# r = subprocess.run(["curl","-s","-o","/dev/null","-w","%{http_code}","--max-time","5","https://huggingface.co"],
#                    capture_output=True, text=True)
# print(f"huggingface.co:  HTTP {r.stdout or 'timeout'}")

# # 3) Ищем все Qwen-модели от unsloth (чтобы найти точное имя)
# from huggingface_hub import HfApi, hf_hub_download
# api = HfApi(token=os.environ.get("HF_TOKEN"))
# print("\nQwen-модели от unsloth (фильтр по имени):")
# found = []
# for m in api.list_models(author="unsloth", search="Qwen2.5", cardData=False):
#     print(f"  {m.id}")
#     found.append(m.id)
# if not found:
#     print("  (ни одной не найдено — возможно, нужен токен)")

# # 4) Проверяем что config.json скачивается с токеном
# target = "unsloth/Qwen2.5-4B-Instruct-bnb-4bit"
# print(f"\nПробуем скачать config.json из {target}:")
# try:
#     path = hf_hub_download(target, "config.json", token=os.environ.get("HF_TOKEN"))
#     print(f"  ✓ скачан: {path}")
# except Exception as e:
#     print(f"  ✗ {e}")

# # 5) Место на диске
# total, used, free = shutil.disk_usage("/")
# print(f"\ndisk: total={total//1e9:.0f}GB  used={used//1e9:.0f}GB  free={free//1e9:.0f}GB")


In [ ]:
from unsloth import FastLanguageModel

In [ ]:

MAX_SEQ_LEN = 4096

# 3B-Instruct: ~7 GB VRAM peak на T4, полный train ~1 час.
# На нашем датасете (~800 примеров) разница в качестве с 7B небольшая.
# Если захочешь вернуться к 7B — замени на "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
# и верни per_device_train_batch_size=1, gradient_accumulation_steps=8.

model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print(f"tokenizer type: {type(tokenizer).__name__}")
print(f"chat template присутствует: {bool(tokenizer.chat_template)}")


==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer type: Qwen2Tokenizer
chat template присутствует: True


## 3. Wrap в LoRA

**Гиперпараметры — почему именно эти:**
- `r=16` — стандарт для 4B на специализированной задаче. Меньше r — недостаточная capacity, больше — переобучение и лишний VRAM.
- `lora_alpha=32` (= 2×r) — стандартное соотношение. Альфа отвечает за эффективный learning rate LoRA-веток.
- `target_modules` — все 7 проекций attention+MLP. Можно ограничиться только attention (`q,k,v,o`) — будет на 30% быстрее, но качество чуть ниже.
- `lora_dropout=0.05` — лёгкая регуляризация. На больших датасетах (>10k) можно 0.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # экономит VRAM ценой 20% времени
    random_state=42,
)
model.print_trainable_parameters()

Unsloth: Already have LoRA adapters! We shall skip this step.


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 4. Применяем chat template

**Учебный нюанс — почему `apply_chat_template`:**
Каждое семейство моделей имеет свой формат разделителей. Qwen использует ChatML:

```
<|im_start|>system
...
<|im_end|>
<|im_start|>user
...
<|im_end|>
<|im_start|>assistant
...
<|im_end|>
```

Если форматировать вручную — легко промахнуться (пропустить `\n`, перепутать токены) и модель учится мусору. `tokenizer.apply_chat_template` гарантированно даёт **тот же** строковой формат, что увидит `tokenizer.encode` на инференсе. Это та самая инвариантность train==inference.

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={
    "train": "train.jsonl",
    "val":   "val.jsonl",
})

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

ds = raw.map(format_chat, remove_columns=["messages", "_meta"])
print(f"train: {len(ds['train'])}, val: {len(ds['val'])}")
print("--- sample formatted text (first 500 chars) ---")
print(ds["train"][0]["text"][:500])

Map:   0%|          | 0/8301 [00:00<?, ? examples/s]

Map:   0%|          | 0/834 [00:00<?, ? examples/s]

train: 8301, val: 834
--- sample formatted text (first 500 chars) ---
<|im_start|>system

Ты экспертный Text2SQL ассистент с глубоким пониманием банковской аналитики.
Твоя задача - точно преобразовывать запросы на русском языке в оптимизированные SQL-запросы.

### Схема базы данных:
TABLE geographic
  - city (TEXT)
  - county (TEXT)
  - region (TEXT)

TABLE generalinfo
  - id_restaurant (BIGINT)
  - label (TEXT)
  - food_type (TEXT)
  - city (TEXT)
  - review (REAL)

TABLE location
  - id_restaurant (BIGINT)
  - street_num (BIGINT)
  - street_name (TEXT)
  - city 


In [ ]:
# # Сколько токенов получается? Это важно — выяснить, не уходим ли за MAX_SEQ_LEN.
# import numpy as np
# # lens = [len(tokenizer.encode(x["text"])) for x in ds["train"].select(range(min(500, len(ds['train']))))]
# sample = ds["train"].select(range(min(500, len(ds["train"]))))
# lens = [len(tokenizer(x["text"])["input_ids"]) for x in sample]
# print(f"token len: p50={np.percentile(lens,50):.0f}  p90={np.percentile(lens,90):.0f}  "
#       f"p99={np.percentile(lens,99):.0f}  max={max(lens)}")
# print(f"будут обрезаны: {sum(1 for l in lens if l > MAX_SEQ_LEN)} / {len(lens)} (sample)")
# # Если >5% обрезается — увеличь MAX_SEQ_LEN до 6144 (если памяти хватает)
# # или сократи db_schema/column_stats в системных промптах



# Сколько токенов получается? Важно убедиться, что не уходим за MAX_SEQ_LEN.
import numpy as np

# Используем tokenizer(text)["input_ids"] вместо tokenizer.encode(text) —
# это работает и для PreTrainedTokenizer, и для быстрых Rust-tokenizers.
sample = ds["train"].select(range(min(500, len(ds["train"]))))
lens = [len(tokenizer(x["text"])["input_ids"]) for x in sample]

print(f"token len: p50={np.percentile(lens,50):.0f}  p90={np.percentile(lens,90):.0f}  "
      f"p99={np.percentile(lens,99):.0f}  max={max(lens)}")
truncated = sum(1 for l in lens if l > MAX_SEQ_LEN)
print(f"будут обрезаны: {truncated} / {len(lens)} ({100*truncated/len(lens):.1f}%)")
# Норма: <5% обрезается. Если больше — увеличь MAX_SEQ_LEN до 6144
# или укороти схему БД в системном промпте (убери column_stats).


token len: p50=3003  p90=6975  p99=107948  max=108018
будут обрезаны: 174 / 500 (34.8%)


## 5. Trainer

**Гиперпараметры обучения:**
- `lr=2e-4` — типично для LoRA. Full FT использует 1e-5; LoRA даёт «дельту» поверх замороженной модели → можно учить агрессивнее.
- `warmup_ratio=0.05` — линейный разгон в первых 5% шагов. Без warmup loss часто взрывается на первых батчах.
- `2 эпохи` — золотая середина. 1 эпоха часто недоучивает, 3+ переобучается на конкретных схемах.
- `effective batch = 8` — `per_device=2 × grad_accum=4`. На T4 нельзя поднять выше per_device=2 при seq=4096.
- `bf16=False` — T4 не поддерживает bfloat16, только fp16. На A100/H100 ставь `bf16=True`.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Чекпоинты пишем прямо в Drive — переживут дисконнект сессии.
CKPT_DIR = "/content/drive/MyDrive/MyDrive/text2sql_finetune/checkpoints"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        output_dir=CKPT_DIR,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch = 8
        warmup_ratio=0.05,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,                       # A100/H100: bf16 точнее fp16 и быстрее
        logging_steps=10,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=5,
        eval_strategy="steps",
        eval_steps=100,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        packing=True,
        report_to="none",
        seed=42,
    ),
)
print(f"Чекпоинты будут сохраняться в: {CKPT_DIR}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/8301 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=16):   0%|          | 0/8301 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/834 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=16):   0%|          | 0/834 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Чекпоинты будут сохраняться в: /content/drive/MyDrive/MyDrive/text2sql_finetune/checkpoints


In [ ]:
# # Smoke-train: 20 шагов на маленьком сабсэмпле.
# # Цель: убедиться что loss падает (а не NaN / взрывается) до запуска полного прогона.
# import copy

# smoke_args = copy.deepcopy(trainer.args)
# smoke_args.max_steps = 20
# smoke_args.eval_strategy = "no"   # ← меняем ДО SFTTrainer.__init__
# smoke_args.save_strategy = "no"   #   иначе он проверяет eval_dataset и падает

# smoke_trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=ds["train"].select(range(min(64, len(ds["train"])))),
#     args=smoke_args,
# )
# smoke_trainer.train()
# Ожидаем: loss ~1.5-3.0 в начале, плавно падает к ~0.5-1.5 за 20 шагов.
# NaN или рост loss → проблема с chat-template или dtype.



# Unsloth: Tokenizing ["text"] (num_proc=6): 100%
#  64/64 [00:16<00:00,  4.30 examples/s]
# The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
# ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
#    \\   /|    Num examples = 64 | Num Epochs = 3 | Total steps = 20
# O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
# \        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
#  "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
# `use_return_dict` is deprecated! Use `return_dict` instead!
# Unsloth: Will smartly offload gradients to save VRAM!
# Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
#  [20/20 13:23, Epoch 2/3]
# Step	Training Loss
# 10	1.139642
# 20	0.853441
# TrainOutput(global_step=20, training_loss=0.9965413093566895, metrics={'train_runtime': 858.207, 'train_samples_per_second': 0.186, 'train_steps_per_second': 0.023, 'total_flos': 9020510927142912.0, 'train_loss': 0.9965413093566895, 'epoch': 2.5})

In [ ]:
# import shutil
# from pathlib import Path
# shutil.rmtree("/content/drive/MyDrive/text2sql_finetune/checkpoints", ignore_errors=True)
# print("✓ чекпоинты удалены")

In [ ]:
from pathlib import Path

# Автодетект последнего чекпоинта — если сессия упала, продолжаем с места остановки.
ckpt_dir = Path(CKPT_DIR)
last_ckpt = None
if ckpt_dir.exists():
    ckpts = sorted(
        [d for d in ckpt_dir.iterdir() if d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[1])
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"↻ Resuming from {last_ckpt}")
    else:
        print("▶ Чекпоинтов нет — старт с нуля")
else:
    print("▶ Папка чекпоинтов не найдена — старт с нуля")

trainer.train(resume_from_checkpoint=last_ckpt)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


↻ Resuming from /content/drive/MyDrive/MyDrive/text2sql_finetune/checkpoints/checkpoint-795


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,360 | Num Epochs = 1 | Total steps = 795
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss,Validation Loss


TrainOutput(global_step=795, training_loss=0.0, metrics={'train_runtime': 0.0074, 'train_samples_per_second': 863265.701, 'train_steps_per_second': 107908.213, 'total_flos': 1.0503109699655731e+18, 'train_loss': 0.0, 'epoch': 1.0})

In [ ]:
# Сохраняем LoRA-адаптер локально (~165 MB)
model.save_pretrained("outputs/qwen25-Coder-7B-text2sql-lora/adapter")
tokenizer.save_pretrained("outputs/qwen25-Coder-7B-text2sql-lora/adapter")
!ls -la outputs/qwen25-Coder-7B-text2sql-lora/adapter/
!du -sh outputs/qwen25-Coder-7B-text2sql-lora/adapter/

# Бэкап в Drive — переживёт дисконнект сессии
import shutil
DRIVE_ADAPTER = "/content/drive/MyDrive/MyDrive/text2sql_finetune/adapter"
shutil.copytree(
    "outputs/qwen25-Coder-7B-text2sql-lora/adapter",
    DRIVE_ADAPTER,
    dirs_exist_ok=True,
)
print(f"✓ адаптер сохранён в Drive: {DRIVE_ADAPTER}")

## 6. Quick eval — посмотреть, как модель отвечает

Перед тем как мерджить и конвертировать — ручная проверка на 2-3 примерах из val. Если модель отвечает мусором или не следует формату — что-то не так с chat template / training format.

In [ ]:
FastLanguageModel.for_inference(model)  # переключает unsloth в режим инференса (ускорение)

import json
raw_val = [json.loads(l) for l in open("val.jsonl")][:3]

for ex in raw_val:
    msgs = ex["messages"][:2]  # без assistant
    inputs = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to("cuda")
    out = model.generate(inputs, max_new_tokens=256, do_sample=False, temperature=0.0)
    pred = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    gold = ex["messages"][2]["content"]
    print(f"--- {ex.get('_meta', {})} ---")
    print(f"PRED: {pred[:300]}")
    print(f"GOLD: {gold[:300]}\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

--- {'db_id': 'superhero', 'question_id': 768, 'task': 'sql', 'source': 'train_queries_pg.json'} ---
PRED: SELECT COUNT(T1.superhero_name) FROM superhero AS T1 INNER JOIN publisher AS T2 ON T1.publisher_ID = T2.ID WHERE T2.publisher_name = 'Dark Horse Comics'
GOLD: SELECT COUNT(T1.id) FROM superhero AS T1 INNER JOIN publisher AS T2 ON T1.publisher_id = T2.id WHERE T2.publisher_name = 'Dark Horse Comics'



Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- {'db_id': 'superhero', 'question_id': 777, 'task': 'sql', 'source': 'train_queries_pg.json'} ---
PRED: SELECT T FROM (SELECT CASE WHEN T2.superhero_name = 'Agent 13' THEN T1.gender END AS T FROM gender AS T1 INNER JOIN superhero AS T2 ON T1.ID = T2.gender_id) WHERE NOT T IS NULL
GOLD: SELECT T2.gender FROM superhero AS T1 INNER JOIN gender AS T2 ON T1.gender_id = T2.id WHERE T1.superhero_name = 'Agent 13'

--- {'db_id': 'superhero', 'question_id': 823, 'task': 'sql', 'source': 'train_queries_pg.json'} ---
PRED: SELECT COUNT(*) FROM superhero AS T1 INNER JOIN publisher AS T2 ON T1.publisher_id = T2.ID INNER JOIN gender AS T3 ON T1.gender_id = T3.ID WHERE T2.publisher_name = 'Marvel Comics' AND T3.gender = 'Female'
GOLD: SELECT COUNT(T1.id) FROM superhero AS T1 INNER JOIN publisher AS T2 ON T1.publisher_id = T2.id INNER JOIN gender AS T3 ON T1.gender_id = T3.id WHERE T2.publisher_name = 'Marvel Comics' AND T3.gender = 'Female'



## 7. Merge LoRA → HF safetensors → GGUF q4_K_M

**Учебный нюанс — формат GGUF:**
- `q4_K_M` — 4-bit с K-quants и Medium-блоками. Лучший trade-off для 4B: ~3 GB файл, потеря качества <1% относительно fp16.
- `q5_K_M` — 5-bit, ~3.5 GB, потери почти 0. Если место не критично — бери.
- `q8_0` — 8-bit, ~4.5 GB. Эталон, минимальные потери. Лучше для финальной публикации.
- `q3_K_S` — 3-bit, ~2 GB. Заметно деградирует на сложных SQL. Не бери для production.

Конвертация ~5 минут на 4B.

In [ ]:
from unsloth import FastLanguageModel
import glob, shutil
from pathlib import Path

# Путь к адаптеру, сохранённому ячейкой выше (переживает дисконнект)
ADAPTER_PATH = "/content/drive/MyDrive/MyDrive/text2sql_finetune/adapter"
DRIVE_DIR    = "/content/drive/MyDrive/MyDrive/text2sql_finetune"

# Загружаем базовую модель + LoRA-адаптер из Drive
# (from_pretrained читает adapter_config.json → base_model_name_or_path → тянет Qwen с HF)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

# Merge LoRA → full weights → GGUF q4_K_M
# unsloth создаёт папку с суффиксом _gguf, поэтому ищем glob-ом
model.save_pretrained_gguf(
    "qwen25-Coder-7B-text2sql",
    tokenizer,
    quantization_method="q4_k_m",
)

# Копируем все найденные GGUF в Drive, пропускаем дубли
gguf_files = [f for f in glob.glob("/content/**/*.gguf", recursive=True)
              if "/content/drive/" not in f]
print(f"Найдено GGUF: {gguf_files}")

for f in gguf_files:
    dst = Path(DRIVE_DIR) / Path(f).name
    if dst.exists():
        print(f"⚠ уже есть, пропускаем: {dst.name}")
        continue
    shutil.copy(f, dst)
    print(f"✓ сохранён в Drive: {dst.name}  ({dst.stat().st_size / 1e9:.2f} GB)")

In [ ]:
# Шаг 1: проверяем что GGUF существует
import glob
files = glob.glob("qwen25-Coder-7B-text2sql/*.gguf") + glob.glob("qwen25-Coder-7B-text2sql/*.gguf")
print("GGUF файлы:", files)

In [ ]:
import glob, shutil
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/MyDrive/text2sql_finetune"

# unsloth добавляет суффикс _gguf к папке — ищем glob-ом, исключаем сам Drive
gguf_files = [f for f in glob.glob("/content/**/*.gguf", recursive=True)
              if "/content/drive/" not in f]

if not gguf_files:
    print("✗ GGUF не найден — сначала запусти ячейку с save_pretrained_gguf")
else:
    for f in gguf_files:
        size_gb = Path(f).stat().st_size / 1e9
        dst = Path(DRIVE_DIR) / Path(f).name
        print(f"Размер: {size_gb:.2f} GB  →  {dst}")
        if dst.exists():
            print(f"⚠ уже есть, пропускаем")
            continue
        shutil.copy(f, dst)
        print(f"✓ сохранён в Drive: {dst.name}")

In [ ]:
# unsloth helper делает merge + GGUF в одной операции
model.save_pretrained_gguf(
    "qwen25-Coder-7B-text2sql",
    tokenizer,
    quantization_method="q4_k_m",
)
!ls -la qwen25-Coder-7B-text2sql/

In [ ]:
# Скачать GGUF локально (или сохранить в Drive)
!cp qwen25-Coder-7B-text2sql/*.gguf /content/drive/MyDrive/text2sql_finetune/ 2>/dev/null && echo 'saved to Drive' || echo 'no Drive'
# либо через File browser → Download

## 8. Что дальше

1. Скачай GGUF локально на Mac.
2. Положи в `~/.lmstudio/models/local/qwen35-text2sql/qwen35-text2sql.q4_k_m.gguf` (создай папку — LM Studio её увидит).
3. В LM Studio: открой модель → Local Server → Start.
4. В `.env` проекта переключи:
   ```
   LLM_BASE_URL=http://localhost:1234/v1
   LLM_API_KEY=lm-studio
   LLM_MODEL_NAME=qwen35-text2sql
   ```
5. `./scripts/run_ablation.sh feat/lora_finetune` — eval на bird_small + ambrosia_small.

См. `scripts/lm_studio_smoke_test.py` для проверки локального API.